## Dashboard de Clima en Tiempo Real

En esta actividad vamos a construir un pequeño dashboard interactivo usando:

- La API de OpenWeatherMap para obtener datos climáticos actualizados.
- Python para hacer las solicitudes HTTP y procesar los resultados.
- Dash para visualizar la información en una página web local e interactiva.

Nuestro objetivo será permitir al usuario ingresar una ciudad y visualizar en tiempo real el estado del clima actual en esa ubicación.


In [1]:
!pip install dash dash-bootstrap-components requests dash-leaflet


In [2]:
import os
from getpass import getpass
import requests

In [3]:
os.environ["API_KEY"] = getpass("Ingresa tu API KEY de OpenWeatherMap: ") # Regístrate gratis en https://openweathermap.org/api
API_KEY = os.getenv("API_KEY")

Ingresa tu API KEY de OpenWeatherMap: ··········


In [4]:
def get_clima(ciudad, api_key):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={ciudad}&appid={api_key}&units=metric&lang=es"
    r = requests.get(url)
    if r.status_code != 200:
        return None
    return r.json()

In [5]:
data = get_clima("Santiago", API_KEY)
data["main"], data["weather"][0]

({'temp': 17.54,
  'feels_like': 16.68,
  'temp_min': 16.14,
  'temp_max': 18.94,
  'pressure': 1017,
  'humidity': 51,
  'sea_level': 1017,
  'grnd_level': 931},
 {'id': 804, 'main': 'Clouds', 'description': 'nubes', 'icon': '04d'})

In [6]:
import dash
from dash import html, dcc, Output, Input, State
import dash_bootstrap_components as dbc
import dash_leaflet as dl
import requests

In [7]:
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
app.title = "Clima en tiempo real"

In [8]:
app.layout = dbc.Container([
    html.H2("Clima actual por ciudad", className="my-4"),

    dbc.Row([
        dbc.Col(dcc.Input(id="ciudad-input", type="text", placeholder="Ej. Santiago", debounce=True, className="form-control"), width=6),
        dbc.Col(html.Button("Consultar", id="consultar-btn", n_clicks=0, className="btn btn-primary"), width="auto")
    ], className="mb-4"),

    dcc.Tabs(id="tabs", value="info", children=[
        dcc.Tab(label="Información del clima", value="info"),
        dcc.Tab(label="Mapa", value="mapa")
    ]),

    dcc.Store(id="datos-clima"),
    html.Div(id="contenido-tabs")
])

In [9]:
# Callback para obtener datos y forzar pestaña activa
@app.callback(
    Output("datos-clima", "data"),
    Input("consultar-btn", "n_clicks"),
    State("ciudad-input", "value"),
    prevent_initial_call=True
)
def obtener_datos(n, ciudad):
    if not ciudad:
        return dash.no_update
    datos = get_clima(ciudad, API_KEY)
    return datos if datos else {}

def obtener_datos_y_mantener_tab(n, ciudad, current_tab):
    if not ciudad:
        return dash.no_update, current_tab
    datos = get_clima(ciudad, API_KEY)
    # Mantenemos la pestaña activa actual, así la información se actualiza en esa misma
    return datos if datos else {}, current_tab

# Callback para mostrar contenido según pestaña activa
@app.callback(
    Output("contenido-tabs", "children"),
    Input("tabs", "value"),
    Input("datos-clima", "data")
)
def mostrar_contenido(tab, datos):
    if not datos or "weather" not in datos:
        return dbc.Alert("Ingrese una ciudad y presione Consultar.", color="info")

    ciudad = datos.get("name", "Ciudad")
    clima = datos["weather"][0]["description"]
    temp = datos["main"]["temp"]
    humedad = datos["main"]["humidity"]
    viento = datos["wind"]["speed"]
    lat = datos["coord"]["lat"]
    lon = datos["coord"]["lon"]

    if tab == "info":
        return dbc.Card([
            dbc.CardBody([
                html.H4(ciudad, className="card-title"),
                html.P(f"Condición: {clima.capitalize()}"),
                html.P(f"Temperatura: {temp}°C"),
                html.P(f"Humedad: {humedad}%"),
                html.P(f"Viento: {viento} m/s")
            ])
        ], style={"width": "22rem"})

    elif tab == "mapa":
        return dl.Map(center=[lat, lon], zoom=10, children=[
            dl.TileLayer(),
            dl.Marker(position=[lat, lon], children=[dl.Tooltip(ciudad.title())])
        ], style={"width": "100%", "height": "400px"})

    return html.Div()

In [10]:
if __name__ == "__main__":
    app.run(debug=True)

<IPython.core.display.Javascript object>

Este ejemplo muestra cómo combinar Python, APIs y Dash para construir una aplicación web interactiva. La clave está en:
- Recuperar información en tiempo real usando requests.
- Procesarla de forma clara.
- Presentarla con una interfaz amigable para el usuario.
